# Best-response red attackers on CAGE-2
For every frozen defender, a fresh recurrent-PPO red is trained from scratch against it under one shared budget, following the approximate-exploitability protocol of Timbers et al. (2022). These attackers are the fourth member of the evaluation suite in `eval.ipynb`.

- **Attacker interface**, identical for every defender: a GRU over [red's private state, previous action one-hot], conditioned on the full $\theta$. It maximizes the negative of the physical payoff `Reward2`, the evaluation payoff.
- **Head start**: as in `eval.ipynb`, red's $h$ head-start steps are played by the trained red for every attacker.
- **Defenders**: `Critic P1` (the Oracle), `Trained P1` (our method), `RPPO`, `DR_RPPO`, `VARIBAD`, and `MMD`. Each defender samples its action from its own distribution, as at deployment.
- **Output**: `compared/BR_attacker_<defender>.pth`, holding the weights, the configuration, and the defender's return per update.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
BR_TOTAL_EPISODES = 1000     # same budget for every defender
EPISODES_PER_UPD  = 10
GAMMA             = 0.98
GAE_LAMBDA        = 0.95
PPO_EPOCHS        = 4
CLIP_EPS          = 0.2
ENT_COEF          = 0.01
VF_COEF           = 0.5
LR                = 3e-4
CKPT_EVERY        = 20
SEED_BASE         = 777
SAVE_DIR          = "compared"
os.makedirs(SAVE_DIR, exist_ok=True)

## Frozen defenders (the wrappers are defined in `Initialize.ipynb`)

In [ ]:
Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(Player1Actor().to(device), "player1.pth")
Player2       = Load(AttackerActor().to(device), "player2.pth")
trained_red   = red_policy(Player2)
BASELINES     = Load_Baselines(SAVE_DIR)

print("Loaded baselines:", list(BASELINES.keys()))
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'MMD']

## Rollouts and PPO (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
def gae(rews, vals):
    T = len(rews)
    adv = np.zeros(T, dtype=np.float32); last = 0.0
    for t in reversed(range(T)):
        nv = vals[t + 1] if t + 1 < T else 0.0
        delta = rews[t] + GAMMA * nv - vals[t]
        last = delta + GAMMA * GAE_LAMBDA * last
        adv[t] = last
    return adv, adv + np.asarray(vals, dtype=np.float32)


def rollout_br(attacker, defw):
    Theta, Type_Theta, w, p_exploit, h = Sample_Context()
    blue_state, _, red_state, red_knowledge, real_state = Reset_Episode()
    red_knowledge, red_state, real_state = head_start(trained_red, h, red_knowledge, red_state, Type_Theta, p_exploit)
    defw.reset(blue_state, Theta, Type_Theta)
    theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
    hdn = None
    u_prev = np.zeros(n_u, dtype=np.float32)
    o_l, u_l, lp_l, v_l, ar_l, dr_l = [], [], [], [], [], []
    for t in range(max_steps):
        a_dist = defw.act_dist(red_state, real_state)
        a = int(np.random.choice(n_a, p=a_dist))            # the defender samples from its own policy
        obs = np.concatenate([red_state, u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, val, hdn = attacker(torch.tensor(obs, device=device).view(1, 1, -1), theta_t, hdn)
        dist = Categorical(logits=logits.view(-1))
        u = int(dist.sample())
        lp = float(dist.log_prob(torch.tensor(u, device=device)))
        _, red_knowledge, red_state, _ = red_step(u, red_knowledge, red_state, p_exploit)
        blue_action_name, y, real_state = blue_step(a)
        defw.observe(a, blue_action_name, y, t)
        r = Reward2(real_state, w)
        o_l.append(obs); u_l.append(u); lp_l.append(lp); v_l.append(val.item())
        ar_l.append(-r); dr_l.append(r)                      # red maximizes -r
        u_prev = np.zeros(n_u, dtype=np.float32); u_prev[u] = 1.0
    adv, ret = gae(np.array(ar_l, dtype=np.float32), np.array(v_l, dtype=np.float32))
    return {'obs': np.stack(o_l), 'acts': np.array(u_l), 'logp': np.array(lp_l, dtype=np.float32),
            'adv': adv, 'ret': ret, 'theta': Theta.astype(np.float32), 'def_rews': np.array(dr_l, dtype=np.float32)}


def ppo_update_br(att, opt, eps):
    S = lambda k, dt=torch.float32: torch.tensor(np.stack([e[k] for e in eps]), dtype=dt, device=device)
    obs, acts, oldlp, adv, ret = S('obs'), S('acts', torch.long), S('logp'), S('adv'), S('ret')
    theta = S('theta').unsqueeze(1).expand(-1, max_steps, -1)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values, _ = att(obs, theta)
        dist = Categorical(logits=logits)
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv, torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(att.parameters(), 1.0)
        opt.step()


def train_br(def_name, seed):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    defw = get_defender(def_name)
    att = BRAttacker().to(device)
    opt = optim.AdamW(att.parameters(), lr=LR)
    n_upd = BR_TOTAL_EPISODES // EPISODES_PER_UPD
    def_hist = []
    fname = 'BR_attacker_' + def_name.replace(' ', '_') + '.pth'
    for upd in tqdm(range(n_upd), desc='BR vs ' + def_name):
        eps = [rollout_br(att, defw) for _ in range(EPISODES_PER_UPD)]
        ppo_update_br(att, opt, eps)
        def_hist.append(float(np.mean([e['def_rews'].sum() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'attacker': att.state_dict(), 'defender': def_name,
                        'config': {'RNN_H': RNN_H, 'seed': seed, 'budget_episodes': BR_TOTAL_EPISODES},
                        'defender_return_per_episode': def_hist},
                       os.path.join(SAVE_DIR, fname))
    return att, def_hist

## Train one attacker per defender

In [ ]:
BR_ATTACKERS, BR_CURVES = {}, {}
for k, name in enumerate(DEFENDERS):
    BR_ATTACKERS[name], BR_CURVES[name] = train_br(name, seed=SEED_BASE + k)

## Convergence check: each defender's return should plateau, otherwise increase `BR_TOTAL_EPISODES`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, c in BR_CURVES.items():
    ax.plot(np.arange(1, len(c) + 1), c, label=name)
ax.set_xlabel('PPO update'); ax.set_ylabel("defender's return per episode")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig('br_curves.jpg', dpi=300, bbox_inches='tight'); plt.show()